# Predict effect of variants

Examples of scoring variant effects with the vinson model. The variants are placed in the GATA motif and CACCC box of the BCL11A erythroid enhancer highlighted in 3.Interpretation.ipynb.<br>
Use cases:
- To check whether a variant changes accessibility, predict density for ref and alt alleles
- To find cell types where variants act, score a table of variants across samples
- To score variants in an individual's sequence context, inject genotypes

## Imports

In [10]:
from io import StringIO
from IPython.display import display

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from genome_tools.genomic_interval import GenomicInterval
from genome_tools.data.extractors import FastaExtractor
from genome_tools.data.anndata import read_zarr_backed
from vinson.utils.sequence_utils import one_hot_encode
from vinson.utils.data_formatting.container import VariantVinsonData
from vinson.datasets.variant import InferenceVariantEmbedDataset
from vinson.inference import score_variants_with_dhs_model
from vinson.utils.helpers import replace_at
from vinson.from_config import read_configs, dhs_model_from_config


# Load auxiliary files
Files are read from public URLs. Download them locally to speed up inference and plotting.

In [2]:
fasta_file = 'https://resources.altius.org/~sabramov/collaborations/vinson/examples/GRCh38_no_alts.fa'

### Note: lazy loading of the zarr anndata is slow outside of the US. Download it locally to speed it up.

In [3]:
zarr_anndata_path = 'https://regulome.altius.org/data/release_20260818/human/consensus_dhs/human_reference/human_reference_anndata.zarr/'
#zarr_anndata_path = '/your/local/path'

In [4]:
# read_zarr_backed function works with python versions > 3.8. For python >=3.11 you can use ad.read_lazy()
human_anndata = read_zarr_backed(zarr_anndata_path)
human_anndata

AnnData object with n_obs × n_vars = 4159 × 5331170
    obs: 'core_annotation', 'extended_annotation', 'core_ontology_term', 'sample_type', 'system', 'subsystem', 'organ', 'organ_region', 'side_position', 'pathological_state', 'germ_layer', 'read_type', 'donor_age', 'donor_age_units', 'donor_ethnicity', 'donor_health_condition', 'donor_id', 'donor_sex', 'growth_stage', 'genotype_id', 'sample_source', 'dnase_units', 'harvest_time', 'harvest_time_units', 'library_date', 'library_method', 'tissue_preparation', 'tissue_purification', 'core_treatment_name', 'treatment1_name', 'treatment1_type', 'treatment1_concentration', 'treatment2_name', 'treatment2_type', 'treatment2_concentration', 'treatment3_name', 'treatment3_type', 'treatment3_concentration', 'treatment4_name', 'treatment4_type', 'treatment4_concentration', 'treatment5_name', 'treatment5_type', 'treatment5_concentration', 'ENCODE_experiment_accession', 'ENCODE_biosample_accession', 'ENCODE_library_accession', 'ENCODE_modification_a

# Load the model

In [5]:
checkpoint_path = '/net/seq/data2/projects/ENCODE4Plus/REGULOME/sequence_to_accessibility_model/dnase_legnet/2026JUN29_8mln/9epochs_8mln_JUN29_best_epoch=8.ckpt'
config_path = '/net/seq/data2/projects/ENCODE4Plus/REGULOME/sequence_to_accessibility_model/dnase_legnet/2026JUN29_8mln/JUN29_8mln_long.yaml'

In [6]:
## set up model
device = 'cpu'
config = read_configs(config_path)
model = dhs_model_from_config(config, checkpoint_path = checkpoint_path).eval().to(device) #L.LightningModule

# Predict effect of a single variant

**Input:** a variant and a sample<br>
**Output:** predicted density for ref and alt alleles

In [7]:
## variant in the middle of the 1344 bp model input
seqlen = 1344
interval = GenomicInterval.from_ucsc('chr2:60495265-60495266').resize(seqlen)
fasta_extr = FastaExtractor(fasta_file)
dna_seq_ref = fasta_extr[interval].upper()

## select sample and its embedding
sample_id = 'AG95417' # HUDEP-2
embed = human_anndata.obsm['motif_embeddings'].loc[sample_id].values

## SNP: G>T, disrupts the GATA motif (TGATAA)
mid = seqlen // 2
print('Reference allele:', dna_seq_ref[mid])
dna_seq_alt = replace_at(dna_seq_ref, mid, 'T')

## one-hot encode and add batch dimension
ohe_seq_ref = torch.as_tensor(one_hot_encode(dna_seq_ref), device=device, dtype=torch.float32)[None,:, :]
ohe_seq_alt = torch.as_tensor(one_hot_encode(dna_seq_alt), device=device, dtype=torch.float32)[None,:, :]

embed = torch.as_tensor(embed, device=device, dtype=torch.float32)[None,:]

## predict
pred_ref, pred_alt = model(ohe_seq_ref, embed), model(ohe_seq_alt, embed)
print('Predictions for ref and alt alleles:', pred_ref.item(), pred_alt.item())

Reference allele: G


/home/sabramov/miniconda3/envs/pytorch/lib/python3.10/site-packages/torch/nn/modules/conv.py:370: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /opt/conda/conda-bld/pytorch_1729647352509/work/aten/src/ATen/native/Convolution.cpp:1036.)
  return F.conv1d(


Predictions for ref and alt alleles: 3.7133352756500244 1.354697346687317


# Predict effects of many variants across cell types

**Input:** a tab-delimited table with columns `chrom, pos, ref, alt, variant_id` and samples<br>
**Output:** ref and alt predictions per variant and sample

Each variant is scored with several shifts of the input window (test-time augmentation); predictions are averaged below.

In [8]:
## variants at core positions of the motifs highlighted in 3.Interpretation.ipynb
## GATA motif (TGATAA): G>T, A>C; CACCC box: C>T, C>T
str_variants = """chrom\tpos\tref\talt\tvariant_id
chr2\t60495266\tG\tT\tchr2_60495266_G_T
chr2\t60495267\tA\tC\tchr2_60495267_A_C
chr2\t60495219\tC\tT\tchr2_60495219_C_T
chr2\t60495220\tC\tT\tchr2_60495220_C_T"""

variants = pd.read_table(StringIO(str_variants))
variants

,chrom,pos,ref,alt,variant_id
0,chr2,60495266,G,T,chr2_60495266_G_T
1,chr2,60495267,A,C,chr2_60495267_A_C
2,chr2,60495219,C,T,chr2_60495219_C_T
3,chr2,60495220,C,T,chr2_60495220_C_T


In [11]:
## samples and shifts to score
shifts = [-10, 0, 10]
selected_samples = ['AG95417', 'AG80853'] # HUDEP-2 and GM12878

## VinsonData for inference
loaded_data = VariantVinsonData.inference_from_variant(variant_df=variants,
                                             sample_ids=selected_samples,
                                             embeddings_df=human_anndata.obsm['motif_embeddings'],
                                             shifts=shifts,
                                             revcomp=False)

## set up dataloader to score
dataset = InferenceVariantEmbedDataset(loaded_data, fasta_file=fasta_file, strict_ref_check=True)
dataloader_kwargs = dict(batch_size=256, 
                        num_workers=2, 
                        pin_memory=True, 
                        shuffle=False)
dataloader = DataLoader(dataset, **dataloader_kwargs)

In [12]:
(alt_scores, ref_scores) = score_variants_with_dhs_model(dataloader, model, device=device)
## get meta for predictions from VinsonData
meta_out = loaded_data.to_df()
meta_out['vinson_ref_score'] = ref_scores
meta_out['vinson_alt_score'] = alt_scores
display(meta_out.head())

/home/sabramov/miniconda3/envs/pytorch/lib/python3.10/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
100%|███████████████████████████████████████████████████████████████████████████████| 1/1 [02:46<00:00, 166.13s/it]


,chrom,pos,ref,alt,sample_id,shift,vinson_ref_score,vinson_alt_score
0,chr2,60495266,G,T,AG95417,-10,1.673828,4.796875
1,chr2,60495266,G,T,AG95417,0,1.575195,4.097656
2,chr2,60495266,G,T,AG95417,10,1.308594,3.281250
3,chr2,60495266,G,T,AG80853,-10,0.021057,0.019653
4,chr2,60495266,G,T,AG80853,0,0.022110,0.021225


The raw table has one row per variant, sample and shift.

## Aggregate predictions across shifts

In [14]:
## geometric mean across shifts
score_cols = ['vinson_ref_score', 'vinson_alt_score']
aggr_df = meta_out.groupby(['chrom', 'pos', 'ref', 'alt', 'sample_id'], observed=True)[score_cols].agg(
    lambda x: np.exp(np.log(x).mean())
).reset_index()
aggr_df['log2_fold_change'] = np.log2(aggr_df['vinson_alt_score'] / aggr_df['vinson_ref_score'])

aggr_df

,chrom,pos,ref,alt,sample_id,vinson_ref_score,vinson_alt_score,log2_fold_change
0,chr2,60495219,C,T,AG80853,0.016998,0.015366,-0.145874
1,chr2,60495219,C,T,AG95417,3.998047,6.417969,0.683105
2,chr2,60495220,C,T,AG80853,0.016678,0.015671,-0.090088
3,chr2,60495220,C,T,AG95417,2.962891,5.621094,0.923828
4,chr2,60495266,G,T,AG80853,0.022217,0.020874,-0.090088
5,chr2,60495266,G,T,AG95417,1.510742,4.011719,1.409180
6,chr2,60495267,A,C,AG80853,0.024124,0.021118,-0.191895
7,chr2,60495267,A,C,AG95417,2.027344,4.906250,1.275391


# Predict variant effects with individual's genotypes

**Input:** a genotype file and `sample_id` -> `indiv_id` correspondence<br>
**Output:** ref and alt predictions on the individual's sequence<br>

To inject genotypes provide a **tab-delimited, tabix-indexed genotype_file** with the following columns:<br>
```chrom, start, end, ref, alt, indiv_id, gt``` - e.g. generated by bcftools query <br>
where `indiv_id` is an ID of individual in vcf file and `gt` is the genotype (e.g. `0/1`). For phased genotypes add an 8th column `phase_set`. <br>


In [15]:
genotype_file = 'https://resources.altius.org/~sabramov/collaborations/vinson/26JUN29/all_variants_stats.trimmed_cols.bed.gz'

In [16]:
## sample_id -> indiv_id correspondence from the anndata
sample_to_indiv_map = human_anndata.obs[['genotype_id']].rename(
    columns={'genotype_id': 'indiv_id'}
)
sample_to_indiv_map

,indiv_id
sample_id,
AG10028,
AG10029,
AG10031,D_INDIV_0002
AG10032,D_INDIV_0002
AG10033,D_INDIV_0002
...,...
AG70923,
AG72731,D_INDIV_0417
AG4026,D_INDIV_0015


In [18]:
## VinsonData for inference, with indiv_id column
loaded_data = VariantVinsonData.inference_from_variant(variant_df=variants,
                                             sample_ids=selected_samples,
                                             embeddings_df=human_anndata.obsm['motif_embeddings'],
                                             sample_indiv_mapping=sample_to_indiv_map,
                                             shifts=shifts)

## set up dataloader to score
dataset = InferenceVariantEmbedDataset(loaded_data, fasta_file=fasta_file, strict_ref_check=True, genotype_file=genotype_file)
dataloader_kwargs = dict(batch_size=256, 
                        num_workers=2, 
                        pin_memory=True, 
                        shuffle=False)
dataloader = DataLoader(dataset, **dataloader_kwargs)

FileNotFoundError: [Errno 2] No such file or directory: 'https://resources.altius.org/~sabramov/collaborations/vinson/26JUN29/all_variants_stats.trimmed_cols.bed.gz'

In [ ]:
(alt_scores, ref_scores) = score_variants_with_dhs_model(dataloader, model, device=device)
## get meta for predictions from VinsonData
out_df = loaded_data.to_df()
out_df['vinson_ref_score'] = ref_scores
out_df['vinson_alt_score'] = alt_scores
out_df.head()